In [1]:
# Initial Setup and Data Processing
from pyspark.sql import SparkSession
from pyspark.sql import types
from pyspark.sql import functions as F


# Create Spark Session
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("homework") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

25/03/03 17:57:52 WARN Utils: Your hostname, MacBook-Air-Mikita.local resolves to a loopback address: 127.0.0.1; using 192.168.100.99 instead (on interface en0)
25/03/03 17:57:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/03 17:57:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.4


In [4]:
# Define schema
# yellow_schema = types.StructType([
#    types.StructField("VendorID",types.IntegerType(),True),
#    types.StructField("tpep_pickup_datetime",types.TimestampType(),True),
#    types.StructField("tpep_dropoff_datetime",types.TimestampType(),True),
#    types.StructField("passenger_count",types.IntegerType(),True),
#    types.StructField("tpep_pickup_datetime",types.IntegerType(),True),
#    types.StructField("trip_distance",types.DoubleType(),True),
#    types.StructField("RatecodeID",types.IntegerType(),True),
#    types.StructField("store_and_fwd_flag",types.StringType(),True),
#    types.StructField("PULocationID",types.IntegerType(),True),
#    types.StructField("DOLocationID",types.IntegerType(),True),
#    types.StructField("payment_type",types.IntegerType(),True),
#    types.StructField("fare_amount",types.DoubleType(),True),
#    types.StructField("extra",types.DoubleType(),True),
#    types.StructField("mta_tax",types.DoubleType(),True),
#    types.StructField("tip_amount",types.DoubleType(),True),
#    types.StructField("tolls_amount",types.DoubleType(),True),
#    types.StructField("improvement_surcharge",types.DoubleType(),True),
#    types.StructField("total_amount",types.DoubleType(),True),
#    types.StructField("congestion_surcharge",types.DoubleType(),True)
#])

In [5]:
# Read and Repartition Data
year = 2024
month = 10

input_path = f'data/raw/yellow/{year}/{month:02d}/'
output_path = f'data/pq/yellow/{year}/{month:02d}/'

# Read original parquet file
df_yellow = spark.read \
    .parquet(f"{input_path}/yellow_tripdata_{year}-{month:02d}.parquet")

# Repartition and save
df_yellow \
    .repartition(4) \
    .write \
    .mode("overwrite") \
    .parquet(output_path)

In [6]:
# October 15th Trips
oct_15_trips = df_yellow.filter(F.to_date(F.col("tpep_pickup_datetime")) == "2024-10-15")
print(f"Trips on October 15th: {oct_15_trips.count()}")

Trips on October 15th: 128893


In [7]:
# Longest Trip
df_with_duration = df_yellow.withColumn(
    "trip_duration_hours",
    F.round((F.unix_timestamp(F.col("tpep_dropoff_datetime"))-
             F.unix_timestamp(F.col("tpep_pickup_datetime"))) / 3600,2)
)
longest_trip = df_with_duration.agg(F.max("trip_duration_hours")).collect()[0][0]
print(f"Longest trip duration (hours): {longest_trip}")

[Stage 9:============================================>              (3 + 1) / 4]

Longest trip duration (hours): 162.62


In [9]:
# Least Frequent Pickup Zone
zones_df = spark.read.csv('taxi_zone_lookup.csv', header = True)

pickup_zones = df_yellow.groupBy("PULocationID") \
    .count() \
    .join(zones_df, df_yellow.PULocationID == zones_df.LocationID) \
    .orderBy("count") \
    .select("Zone","count")

print("Least frequent pickup zone:")
pickup_zones.show(1)

Least frequent pickup zone:


[Stage 15:>                                                         (0 + 4) / 4]

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
+--------------------+-----+
only showing top 1 row

